In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_synthetic_student_dataset (1).csv to cleaned_synthetic_student_dataset (1).csv
Saving p1_synthetic_test.csv to p1_synthetic_test.csv
Saving p1_synthetic_train.csv to p1_synthetic_train.csv
Saving p1_synthetic_val.csv to p1_synthetic_val.csv
Saving synthetic_student_dataset.csv to synthetic_student_dataset.csv


In [ ]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import cross_val_score

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier


# ===============================
# Load Training Data
# ===============================

train = pd.read_csv('/content/p1_synthetic_train.csv')

train = train.dropna(
    subset=[
        'mastery_score',
        'decay_rate',
        'difficulty',
        'response_correct',
        'bloom_tier'
    ]
)

print("Train Shape:", train.shape)


# ===============================
# Class Distribution
# ===============================

print("\nClass Distribution")
print(train['mastery_score'].value_counts())

print("\nPercentage Distribution")
print(train['mastery_score'].value_counts(normalize=True) * 100)


# ===============================
# Feature Selection
# ===============================
features = [
    'decay_rate',
    'difficulty',
    'response_correct',
    'bloom_tier'
]
X_train = train[features]
y_train = train['mastery_score']


# ===============================
# Apply SMOTE
# ===============================

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE")
print(y_resampled.value_counts())


# ===============================
# Improved Random Forest
# ===============================

balanced_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

balanced_model.fit(
    X_resampled,
    y_resampled
)

print("\nBalanced RF Training Completed")


# ===============================
# Load Validation Data
# ===============================

val = pd.read_csv('/content/p1_synthetic_val.csv')

val = val.dropna(
    subset=[
        'mastery_score',
        'decay_rate',
        'difficulty',
        'response_correct',
        'bloom_tier'
    ]
)

X_val = val[features]
y_val = val['mastery_score']


# ===============================
# Validation Predictions
# ===============================

val_pred = balanced_model.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)

val_f1 = f1_score(
    y_val,
    val_pred
)

print("\nValidation Accuracy:", val_accuracy)
print("Validation F1:", val_f1)


# ===============================
# Classification Report
# ===============================

print("\nClassification Report")
print(classification_report(y_val, val_pred))


# ===============================
# Confusion Matrix
# ===============================

print("\nConfusion Matrix")
print(confusion_matrix(y_val, val_pred))


# ===============================
# Overfitting Analysis
# ===============================

train_pred = balanced_model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

print("\nTrain Accuracy:", train_accuracy)
print("Validation Accuracy:", val_accuracy)


# ===============================
# Cross Validation
# ===============================

cv_scores = cross_val_score(
    balanced_model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("\nCross Validation Scores")
print(cv_scores)

print("\nMean CV F1 Score")
print(cv_scores.mean())


# ===============================
# Tuned XGBoost
# ===============================

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=2,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(X_val)

print("\nTuned XGBoost Results")
print(classification_report(y_val, xgb_pred))


# ===============================
# Save Improved Model
# ===============================

joblib.dump(
    balanced_model,
    "p1_balanced_rf_improved.pkl"
)

print("\nImproved Model Saved Successfully")

Train Shape: (29629, 9)

Class Distribution
mastery_score
1.0    25438
0.0     4191
Name: count, dtype: int64

Percentage Distribution
mastery_score
1.0    85.855074
0.0    14.144926
Name: proportion, dtype: float64

After SMOTE
mastery_score
0.0    25438
1.0    25438
Name: count, dtype: int64

Balanced RF Training Completed

Validation Accuracy: 0.6234072675790467
Validation F1: 0.745427477669077

Classification Report
              precision    recall  f1-score   support

         0.0       0.19      0.50      0.28       913
         1.0       0.89      0.64      0.75      5444

    accuracy                           0.62      6357
   macro avg       0.54      0.57      0.51      6357
weighted avg       0.79      0.62      0.68      6357


Confusion Matrix
[[ 458  455]
 [1939 3505]]

Train Accuracy: 0.6976273245806474
Validation Accuracy: 0.6234072675790467

Cross Validation Scores
[0.83074695 0.82399666 0.82649448 0.82142484 0.82153749]

Mean CV F1 Score
0.8248400819864763

Tuned XG

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import classification_report

probs = balanced_model.predict_proba(X_val)[:,1]

for threshold in [0.4, 0.5, 0.6, 0.7]:

    preds = (probs > threshold).astype(int)

    print("\nThreshold:", threshold)

    print(classification_report(y_val, preds))


Threshold: 0.4
              precision    recall  f1-score   support

         0.0       0.23      0.22      0.22       913
         1.0       0.87      0.87      0.87      5444

    accuracy                           0.78      6357
   macro avg       0.55      0.55      0.55      6357
weighted avg       0.78      0.78      0.78      6357


Threshold: 0.5
              precision    recall  f1-score   support

         0.0       0.19      0.50      0.28       913
         1.0       0.89      0.64      0.75      5444

    accuracy                           0.62      6357
   macro avg       0.54      0.57      0.51      6357
weighted avg       0.79      0.62      0.68      6357


Threshold: 0.6
              precision    recall  f1-score   support

         0.0       0.17      0.77      0.28       913
         1.0       0.91      0.36      0.52      5444

    accuracy                           0.42      6357
   macro avg       0.54      0.57      0.40      6357
weighted avg       0.80   

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Baseline RF",
        "XGBoost",
        "Balanced RF + SMOTE"
    ],
    "Accuracy":[
        0.8267,
        0.8491,
        0.6234
    ],
    "Class0 Recall":[
        0.05,
        0.01,
        0.50
    ]
})

print(results)

                 Model  Accuracy  Class0 Recall
0          Baseline RF    0.8267           0.05
1              XGBoost    0.8491           0.01
2  Balanced RF + SMOTE    0.6234           0.50


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_weighted = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf_weighted.fit(X_train, y_train)

pred = rf_weighted.predict(X_val)

print(classification_report(y_val, pred))

              precision    recall  f1-score   support

         0.0       0.20      0.37      0.26       913
         1.0       0.88      0.75      0.81      5444

    accuracy                           0.70      6357
   macro avg       0.54      0.56      0.54      6357
weighted avg       0.78      0.70      0.73      6357

